In [1]:
import pandas as pd
import re
import json
import time
import os
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

In [3]:
df = pd.read_csv("Information_Systems_article_data.csv")

df.head()
df.columns.tolist()

['URL',
 'Journal_Title',
 'Article_Title',
 'Volume_Issue',
 'Month_Year',
 'Abstract',
 'Keywords',
 'Author_name',
 'Author_email',
 'Author_Address']

In [4]:
df_clean = pd.DataFrame()

df_clean["URL"] = df["URL"]
df_clean["Journal_Title"] = df["Journal_Title"]

df_clean["Standardized_Title"] = (
    df["Article_Title"]
    .astype(str)
    .str.lower()
    .str.strip()
)

df_clean["month_year"] = df["Month_Year"]

df_clean["Abstract"] = df["Abstract"]
df_clean["Keywords"] = df["Keywords"]

df_clean["Author_Name"] = df["Author_name"]

In [5]:
df_clean["month_year"] = df_clean["month_year"].astype(str)

df_clean["month_year"] = df_clean["month_year"].apply(
    lambda x: re.search(r"\d{4}", x).group() if re.search(r"\d{4}", x) else None
)

df_clean["month_year"] = pd.to_numeric(df_clean["month_year"], errors="coerce")

In [6]:
def clean_author(name):
    if pd.isna(name):
        return None
    
    name = name.strip()
    name = name.replace(".", "")
    name = re.sub(r"[()]", "", name)
    name = re.sub(r"\s+", " ", name)
    
    parts = name.split()
    
    if len(parts) > 1:
        first = parts[0]
        if first.isalpha() and first.isupper() and len(first) <= 3:
            parts[0] = " ".join(list(first))
    
    return " ".join(parts)

df_clean["Standardized_Author"] = df_clean["Author_Name"].apply(clean_author)
df_clean["Standardized_Author"] = df_clean["Standardized_Author"].str.title()

In [7]:
cache_file = "affiliation_cache.json"

try:
    with open(cache_file, "r") as f:
        clean_map = json.load(f)
except:
    clean_map = {}

In [8]:
def clean_address(addr):
    if addr in clean_map:
        return clean_map[addr]
    
    # preprocess multi-affiliation here itself (important)
    addr_clean = addr.strip()
    
    if " and " in addr_clean.lower():
        addr_clean = addr_clean.split(" and ")[-1]
    
    if ";" in addr_clean:
        addr_clean = addr_clean.split(";")[-1]
    
    prompt = f"""
    Extract ONLY ONE primary university, state, and country from the following affiliation.

    Rules:
    - If multiple affiliations exist, use the LAST one.
    - Ignore department names.
    - Ignore street addresses.
    - Use full country names (e.g., "United States").
    
    Input:
    "{addr_clean}"

    Output strictly in JSON:
    {{
        "university": "",
        "state": "",
        "country": ""
    }}
    """
    
    for attempt in range(2):
        try:
            res = client.chat.completions.create(
                model="gpt-4o-mini",
                temperature=0,
                messages=[{"role": "user", "content": prompt}]
            )
            
            content = res.choices[0].message.content.strip()

            # remove markdown formatting
            if "```" in content:
                content = content.split("```")[1]

            # try normal parse
            try:
                output = json.loads(content)
            except:
                start = content.find("{")
                end = content.rfind("}") + 1
                output = json.loads(content[start:end])

            return output
        
        except Exception:
            time.sleep(1)
    
    return {"university": None, "state": None, "country": None}

In [9]:
unique_addresses = df["Author_Address"].dropna().unique()
len(unique_addresses)

2193

In [ ]:
import json
import time

cache_file = "affiliation_cache.json"

# load cache safely
try:
    with open(cache_file, "r") as f:
        clean_map = json.load(f)
except:
    clean_map = {}

for i, addr in enumerate(unique_addresses):

    # skip already processed
    if addr in clean_map:
        continue

    result = clean_address(addr)

    if result["university"] is not None:
        clean_map[addr] = result
    else:
        print(f"Failed: {addr}")

    # save every 25 (more frequent = safer)
    if i % 25 == 0:
        with open(cache_file, "w") as f:
            json.dump(clean_map, f)
        print(f"Saved at {i}/{len(unique_addresses)}")

    time.sleep(0.35)

# final save
with open(cache_file, "w") as f:
    json.dump(clean_map, f)

print("Done!") 

Done!
